# Stage 2: Training YOLOv11n-seg untuk Segmentasi Soft Exudate

**Konfigurasi:**
- Model: YOLOv11n-seg (nano) — cocok untuk dataset kecil & Apple Silicon
- Pre-trained: COCO weights (`yolo11n-seg.pt`)
- Device: MPS (Apple Silicon M1 Pro)
- Dataset: 54 train, 27 val (sangat kecil → augmentasi agresif)
- Class: 1 (soft_exudate) — class imbalance parah

## 1. Cek Environment

In [1]:
import torch
from ultralytics import YOLO
from pathlib import Path

print(f"PyTorch version : {torch.__version__}")
print(f"MPS available   : {torch.backends.mps.is_available()}")
print(f"Device          : {'mps' if torch.backends.mps.is_available() else 'cpu'}")

# Cek file yang dibutuhkan
weights_path = Path("yolo11n-seg.pt")
data_path    = Path("yolo_dataset/data.yaml")

print(f"\nWeights file    : {weights_path} — {'OK' if weights_path.exists() else 'TIDAK ADA'}")
print(f"Data config     : {data_path} — {'OK' if data_path.exists() else 'TIDAK ADA'}")

PyTorch version : 2.10.0
MPS available   : True
Device          : mps

Weights file    : yolo11n-seg.pt — OK
Data config     : yolo_dataset/data.yaml — OK


## 2. Load Model

In [2]:
model = YOLO("yolo11n-seg.pt")
print(f"Model loaded: {model.model_name}")
print(f"Task: {model.task}")

Model loaded: yolo11n-seg.pt
Task: segment


## 3. Training (v2 — imgsz=1024)

**Perubahan dari run sebelumnya:**
- `imgsz=1024` (sebelumnya 640) — SE kecil lebih terlihat di resolusi tinggi
- `batch=4` (sebelumnya 8) — dikecilkan karena imgsz lebih besar, agar muat di memory
- `close_mosaic=30` — matikan mosaic di 30 epoch terakhir agar training lebih stabil di akhir
- `epochs=300, patience=80` — patience dinaikkan, beri waktu lebih lama untuk konvergen

In [5]:
model = YOLO("yolo11n-seg.pt")

results = model.train(
    data="yolo_dataset/data.yaml",
    epochs=300,
    patience=80,
    imgsz=1024,
    batch=4,
    device="mps",

    # === Augmentasi (agresif untuk dataset kecil) ===
    augment=True,
    mosaic=1.0,
    close_mosaic=30,
    mixup=0.15,
    copy_paste=0.15,
    degrees=15.0,
    scale=0.5,
    fliplr=0.5,
    flipud=0.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,

    # === Optimizer ===
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=10,

    # === Lainnya ===
    project="runs",
    name="soft_exudate_seg_v2",
    exist_ok=True,
    save=True,
    save_period=50,
    plots=True,
    verbose=True,
)

Ultralytics 8.4.21 🚀 Python-3.14.0 torch-2.10.0 MPS (Apple M1 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=30, cls=0.5, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/data.yaml, degrees=15.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=soft_exudate_seg_v2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=80, perspective=0.

## 4. Cek Hasil Training

In [6]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Cari folder hasil training v2
run_dir = Path("runs/soft_exudate_seg_v2")
if not run_dir.exists():
    # YOLO kadang taruh di subfolder segment/
    run_dir = Path("runs/segment/runs/soft_exudate_seg_v2")

print(f"Run dir: {run_dir}")

# Tampilkan training curves
results_img = run_dir / "results.png"
if results_img.exists():
    img = Image.open(results_img)
    plt.figure(figsize=(18, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Training Results (v2 — imgsz=1024)")
    plt.show()
else:
    print(f"results.png belum ada di {run_dir}")

# Tampilkan confusion matrix
cm_img = run_dir / "confusion_matrix.png"
if cm_img.exists():
    img = Image.open(cm_img)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix")
    plt.show()

Run dir: runs/segment/runs/soft_exudate_seg_v2


<Figure size 1800x800 with 1 Axes>

<Figure size 800x800 with 1 Axes>

## 5. Evaluasi pada Validation Set

In [7]:
from pathlib import Path
from ultralytics import YOLO

# Cari best.pt
best_path = Path("runs/soft_exudate_seg_v2/weights/best.pt")
if not best_path.exists():
    best_path = Path("runs/segment/runs/soft_exudate_seg_v2/weights/best.pt")

print(f"Best model: {best_path}")
best_model = YOLO(str(best_path))

# Jalankan validasi
metrics = best_model.val(
    data="yolo_dataset/data.yaml",
    device="mps",
    plots=True,
    verbose=True,
)

# Tampilkan metrik utama
print("=" * 50)
print("METRIK EVALUASI v2 (Validation Set)")
print("=" * 50)

print(f"\n[Box Detection]")
print(f"  Precision   : {metrics.box.mp:.4f}")
print(f"  Recall      : {metrics.box.mr:.4f}")
print(f"  mAP50       : {metrics.box.map50:.4f}")
print(f"  mAP50-95    : {metrics.box.map:.4f}")

print(f"\n[Mask Segmentation]")
print(f"  Precision   : {metrics.seg.mp:.4f}")
print(f"  Recall      : {metrics.seg.mr:.4f}")
print(f"  mAP50       : {metrics.seg.map50:.4f}")
print(f"  mAP50-95    : {metrics.seg.map:.4f}")

print(f"\n** Recall = Sensitivity (metrik utama skripsi) **")
print(f"\nPerbandingan dengan v1 (imgsz=640):")
print(f"  v1 → Mask Recall: 0.526, mAP50: 0.709")
print(f"  v2 → Mask Recall: {metrics.seg.mr:.3f}, mAP50: {metrics.seg.map50:.3f}")

Best model: runs/segment/runs/soft_exudate_seg_v2/weights/best.pt
Ultralytics 8.4.21 🚀 Python-3.14.0 torch-2.10.0 MPS (Apple M1 Pro)
YOLO11n-seg summary (fused): 114 layers, 2,834,763 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1777.3±418.7 MB/s, size: 1144.4 KB)
val: Scanning /Users/mac/Documents/Kuliah/Semester 6/Skripsi/yolo_dataset/labels/val.cache... 27 images, 13 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 27/27 14.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 7.0s/it 13.9s23.9s
                   all         27         38      0.751      0.635      0.715      0.401      0.782      0.661      0.751      0.423
Speed: 0.8ms preprocess, 441.2ms inference, 0.0ms loss, 35.1ms postprocess per image
Results saved to /Users/mac/Documents/Kuliah/Semester 6/Skripsi/runs/segment/val
METRIK EVALUASI v2 (Validation Set)

[Box Detecti

## 6. Visualisasi Prediksi pada Validation Set

In [9]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

val_img_dir = Path("yolo_dataset/images/val")
val_lbl_dir = Path("yolo_dataset/labels/val")
gt_se_val   = Path("2. All Segmentation Groundtruths/b. Testing Set/4. Soft Exudates")

# Ambil gambar val yang punya label SE (file label non-kosong)
samples = [
    f.stem for f in sorted(val_lbl_dir.glob("*.txt"))
    if f.stat().st_size > 0
][:6]

# Cari best.pt v2
best_path = Path("runs/soft_exudate_seg_v2/weights/best.pt")
if not best_path.exists():
    best_path = Path("runs/segment/runs/soft_exudate_seg_v2/weights/best.pt")

best_model = YOLO(str(best_path))

fig, axes = plt.subplots(len(samples), 3, figsize=(20, 5 * len(samples)))

for i, stem in enumerate(samples):
    img_path = val_img_dir / f"{stem}.jpg"

    # === Kolom 1: Input (setelah blackout) ===
    orig = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    axes[i, 0].imshow(orig)
    axes[i, 0].set_title(f"{stem} — Input (setelah blackout)")
    axes[i, 0].axis("off")

    # === Kolom 2: Ground Truth SE overlay ===
    gt_path = gt_se_val / f"{stem}_SE.tif"
    if gt_path.exists():
        gt_mask = cv2.imread(str(gt_path), cv2.IMREAD_UNCHANGED)
        if gt_mask.ndim == 3:
            gt_mask = gt_mask[:, :, :3].max(axis=2)
        gt_binary = (gt_mask > 10).astype(np.uint8)

        # Overlay hijau di atas gambar original
        overlay = orig.copy()
        overlay[gt_binary > 0] = [0, 255, 0]
        blended = cv2.addWeighted(orig, 0.7, overlay, 0.3, 0)
        axes[i, 1].imshow(blended)
        n_gt = len(cv2.findContours(gt_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
        axes[i, 1].set_title(f"{stem} — Ground Truth ({n_gt} SE)")
    else:
        axes[i, 1].imshow(orig)
        axes[i, 1].set_title(f"{stem} — Ground Truth (tidak ada)")
    axes[i, 1].axis("off")

    # === Kolom 3: Prediksi ===
    result = best_model.predict(str(img_path), device="mps", verbose=False)[0]
    pred_img = result.plot()
    pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)
    axes[i, 2].imshow(pred_img)
    n_det = len(result.boxes) if result.boxes is not None else 0
    axes[i, 2].set_title(f"{stem} — Prediksi ({n_det} deteksi)")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig("visualisasi_prediksi_v2.png", dpi=100, bbox_inches="tight")
plt.show()
print("Visualisasi disimpan ke visualisasi_prediksi_v2.png")

<Figure size 2000x3000 with 18 Axes>

Visualisasi disimpan ke visualisasi_prediksi_v2.png
